In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

clean_data = pd.read_parquet("sample_by_gene.parquet")
X = clean_data.drop(columns=['Target(SNHG14)', 'sample_type']).values

In [9]:
# Scree plot — inspect to choose N_COMPONENTS in the next cell
n_max = min(len(clean_data), 50)
pca_full = PCA(n_components=n_max)
pca_full.fit(StandardScaler().fit_transform(X))

cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
scree_df = pd.DataFrame({'PC': range(1, n_max + 1), 'Cumulative Variance (%)': cumvar})

fig_scree = px.line(
    scree_df, x='PC', y='Cumulative Variance (%)',
    markers=True,
    title='Scree Plot — Cumulative Explained Variance by Number of PCs'
         '<br><sup>Choose N_COMPONENTS in the next cell based on your desired variance threshold</sup>',
)
fig_scree.add_hline(y=80, line_dash='dash', line_color='orange', annotation_text='80%', annotation_position='right')
fig_scree.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='90%', annotation_position='right')
fig_scree.update_layout(xaxis_title='Number of Principal Components', yaxis_title='Cumulative Variance Explained (%)')
fig_scree.show()

n_80 = int(np.searchsorted(cumvar, 80)) + 1
n_90 = int(np.searchsorted(cumvar, 90)) + 1
print(f'PCs needed for 80% variance: {n_80}')
print(f'PCs needed for 90% variance: {n_90}')

PCs needed for 80% variance: 36
PCs needed for 90% variance: 41


In [10]:
# ── Set N after inspecting the scree plot above ──
N_COMPONENTS = 36

pca = PCA(n_components=N_COMPONENTS)
pcs = pca.fit_transform(StandardScaler().fit_transform(X))

pc_cols = [f'PC{i+1}' for i in range(N_COMPONENTS)]
pca_df = pd.DataFrame(pcs, columns=pc_cols, index=clean_data.index)
pca_df['sample_type'] = clean_data['sample_type'].values

print(f'Using {N_COMPONENTS} PCs — explains {pca.explained_variance_ratio_.sum():.1%} of total variance')

Using 36 PCs — explains 81.1% of total variance


In [11]:
# 2D PCA scatter (PC1 vs PC2) — all treatments, text labels
control_centroid_2d = pca_df[pca_df['sample_type'] == 'Control'][['PC1', 'PC2']].mean()

fig1 = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='sample_type',
    text='sample_type',
    hover_name=pca_df.index,
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    },
    title='PCA of Gene Expression by All Sample Types'
          '<br><sup>Treatments closest to Control in 2D are safest visually; use N-D distance below for full picture</sup>',
)
fig1.update_traces(mode='text')
fig1.update_layout(showlegend=False)

for trace in fig1.data:
    if trace.name == 'Control':
        trace.textfont.color = 'black'
        trace.textfont.size = 14
        trace.textfont.weight = 'bold'
    else:
        trace.textfont.color = trace.marker.color
        trace.textfont.size = 10

fig1.add_trace(go.Scatter(
    x=[control_centroid_2d['PC1']],
    y=[control_centroid_2d['PC2']],
    mode='text',
    text=['⊕ Control Mean'],
    textfont=dict(color='black', size=16, weight='bold'),
    hoverinfo='skip',
    showlegend=False,
))
fig1.show()

In [12]:
# Log2 fold change of Target gene per treatment
target_mean = clean_data.groupby('sample_type')['Target(SNHG14)'].mean()
control_mean = target_mean['Control']
log2fc = np.log2(target_mean / control_mean).drop('Control').sort_values()

fig2 = px.bar(
    log2fc,
    x=log2fc.index,
    y=log2fc.values,
    labels={'x': 'Treatment', 'y': 'Log2 Fold Change vs Control'},
    title='Target Gene (SNHG14) Log2 Fold Change by Treatment (Effectiveness)'
          '<br><sup>Log2FC: −1 = halved (50% reduction), −2 = quartered (75% reduction), 0 = no change relative to control</sup>',
    color=log2fc.values,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
)
fig2.add_hline(y=0, line_dash='dash', line_color='black')
fig2.update_layout(coloraxis_showscale=False, xaxis_tickangle=-45)

tick_log2fc = np.array([-1.2, -1.0, -0.8, -0.6, -0.4, -0.2, 0.0])
tick_pct = (2 ** tick_log2fc - 1) * 100
y_min, y_max = log2fc.min() * 1.15, log2fc.max() * 1.15

fig2.add_trace(go.Scatter(x=[None], y=[None], yaxis='y2', showlegend=False))
fig2.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    yaxis2=dict(
        overlaying='y', side='right', range=[y_min, y_max],
        tickmode='array', tickvals=tick_log2fc,
        ticktext=[f'{p:.0f}%' for p in tick_pct],
        title='% Change vs Control',
    ),
)
fig2.show()

C:\Users\kaian\AppData\Local\Temp\ipykernel_28412\2672145435.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = clean_data.groupby('sample_type')['Target(SNHG14)'].mean()


In [13]:
# N-dimensional Euclidean distance to Control centroid
control_centroid_nd = pca_df[pca_df['sample_type'] == 'Control'][pc_cols].mean()

pca_df['dist_to_control'] = np.sqrt(
    ((pca_df[pc_cols] - control_centroid_nd) ** 2).sum(axis=1)
)

dist_summary = (
    pca_df[pca_df['sample_type'] != 'Control']
    .groupby('sample_type')['dist_to_control']
    .agg(mean_dist='mean', std_dist='std', n='count')
    .sort_values('mean_dist')
)
dist_summary['log2FC'] = log2fc.reindex(dist_summary.index)

fig3 = px.bar(
    dist_summary,
    x=dist_summary.index,
    y='mean_dist',
    error_y='std_dist',
    color='log2FC',
    color_continuous_scale='Reds_r',
    range_color=[log2fc.min(), 0],
    labels={'x': 'Treatment', 'mean_dist': f'Mean Distance to Control Centroid ({N_COMPONENTS} PCs)', 'log2FC': 'Log2FC'},
    title=f'Consistency with Control — Euclidean Distance in {N_COMPONENTS}-PC Space'
          '<br><sup>Bar height = distance to Control (lower = safer); color = log2FC effectiveness (darker red = more effective)</sup>',
)
fig3.update_layout(xaxis_tickangle=-45)
fig3.show()

dist_summary

C:\Users\kaian\AppData\Local\Temp\ipykernel_28412\640477320.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('sample_type')['dist_to_control']


,mean_dist,std_dist,n,log2FC
sample_type,,,,
hATF567,214.203844,3.675084,2,-0.501195
nZF93,216.947575,20.135704,2,-0.415652
Base,218.611365,43.987090,2,-0.119212
hATF561,218.852160,11.123500,2,-0.567325
nZF42,219.829878,56.255625,2,-0.079278
SP1R,224.209301,42.712561,2,0.078562
nZF145,232.058200,52.886760,2,-0.020723
nZF154,232.765651,23.486345,2,-0.313456
hATF555R,242.086856,25.963333,2,-0.009222


In [14]:
# Effectiveness vs Safety tradeoff scatter with Pareto frontier
tradeoff_df = dist_summary.dropna().copy()

sorted_df = tradeoff_df.sort_values('log2FC')
pareto, min_dist = [], float('inf')
for name, row in sorted_df.iterrows():
    if row['mean_dist'] < min_dist:
        pareto.append((row['log2FC'], row['mean_dist']))
        min_dist = row['mean_dist']
pareto_x = [p[0] for p in pareto]
pareto_y = [p[1] for p in pareto]

step_x, step_y = [], []
for i, (fx, fy) in enumerate(zip(pareto_x, pareto_y)):
    if i == 0:
        step_x += [tradeoff_df['log2FC'].min() * 1.05, fx]
        step_y += [fy, fy]
    else:
        step_x += [pareto_x[i - 1], fx]
        step_y += [fy, fy]
step_x.append(fx)
step_y.append(0)

med_log2fc = tradeoff_df['log2FC'].median()
med_dist = tradeoff_df['mean_dist'].median()

fig4 = px.scatter(
    tradeoff_df,
    x='log2FC', y='mean_dist',
    size='std_dist', size_max=25,
    text=tradeoff_df.index,
    color=tradeoff_df.index,
    color_discrete_sequence=px.colors.qualitative.Dark24,
    labels={
        'log2FC': 'Log2 Fold Change (lower = more effective)',
        'mean_dist': f'Mean Distance to Control ({N_COMPONENTS} PCs, lower = safer)',
    },
    title=f'Effectiveness vs Safety Tradeoff ({N_COMPONENTS}-PC Distance)'
          '<br><sup>Bubble size = replicate std dev (larger = less consistent); Pareto frontier = optimal tradeoff candidates</sup>',
)
fig4.update_traces(mode='markers+text', textposition='middle center', textfont=dict(color='black', size=11))

fig4.add_trace(go.Scatter(
    x=step_x, y=step_y, mode='lines',
    line=dict(color='black', width=2, dash='dot'),
    name='Pareto Frontier', showlegend=True,
))

fig4.add_vline(x=med_log2fc, line_dash='dash', line_color='gray', opacity=0.5)
fig4.add_hline(y=med_dist, line_dash='dash', line_color='gray', opacity=0.5)

for x, y, text, color, xanchor, yanchor in [
    (0.01, 0.01, '✓ Safe & Effective',  'green',     'left',  'bottom'),
    (0.99, 0.01, 'Safe but Weak',        'steelblue', 'right', 'bottom'),
    (0.01, 0.99, 'Effective but Risky',  'orange',    'left',  'top'),
    (0.99, 0.99, '✗ Unsafe & Weak',      'red',       'right', 'top'),
]:
    fig4.add_annotation(
        x=x, y=y, text=text,
        xref='paper', yref='paper',
        showarrow=False,
        font=dict(color=color, size=11),
        xanchor=xanchor, yanchor=yanchor,
    )

fig4.update_layout(showlegend=False)
fig4.show()